## Data Preparation

### Purpose
To load, clean, filter, and perform preprocessing the raw Nepali sentiment dataset for use in model training.

### Input
- `sentiment_analysis_nepali_final.csv` — raw labelled Nepali sentiment dataset  
  [https://www.kaggle.com/datasets/aayamoza/nepali-sentiment-analysis] 

### Output
- `covid19_cleaned_sentiments.csv` — COVID-filtered, preprocessed dataset (28,232 rows, 
   positive and negative labels only).
- This file is the input for `model_training.ipynb`.

### Sections
1. Data Cleaning — null check, duplicate resolution
2. Topic Filtering — COVID keyword filtering, core keyword validation
3. Preprocessing — noise removal, stopword removal, rule-based lemmatisation

In [1]:
#Loading the data using pandas dataframe
import pandas as pd
import re

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

#Loading the dataset as a pandas object
df = pd.read_csv("sentiment_analysis_nepali_final.csv")
#df

## Data Cleaning

In [3]:
df.drop("Unnamed: 0", axis=1, inplace = True)

In [4]:
#Checking for null data
df.info()
print('-' * 50)
print(f'\nNo. of null values in data: \n {df.isna().sum()} \n')
print('-' * 50)
print(f'No. of empty sentences in data: {df['Sentences'].isin(['NA', 'N/A', 'null', 'Null', 'None', 'none', 'na', 'n/a']).sum()}')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35789 entries, 0 to 35788
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Sentences  35789 non-null  object
 1   Sentiment  35789 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 559.3+ KB
--------------------------------------------------

No. of null values in data: 
 Sentences    0
Sentiment    0
dtype: int64 

--------------------------------------------------
No. of empty sentences in data: 0


In [5]:
df['Sentences'].str.strip().eq('').sum()

np.int64(0)

In [6]:
#Checking for Duplicate Data
df[df.duplicated(subset = ['Sentences'], keep = False)].sort_values(by=['Sentences'])

,Sentences,Sentiment
947,असली मायामा कचेरा पुछ्दै भिडियोकल गरिन्छ,0
1934,असली मायामा कचेरा पुछ्दै भिडियोकल गरिन्छ,0
28311,अहिले पनि झण्डै डेढ सय आयोजनामा एक लाख हजार कामदार निर्माणस्थलमा रहेको नेपाल निर्माण व्यवयासी महासंघले जनाएको छ कोभिड कोरोना,-1
28327,अहिले पनि झण्डै डेढ सय आयोजनामा एक लाख हजार कामदार निर्माणस्थलमा रहेको नेपाल निर्माण व्यवयासी महासंघले जनाएको छ कोभिड कोरोना,1
226,अहिले सम्मको झल्लो फिल्म राम्रो नि लागेन,-1
18,अहिले सम्मको झल्लो फिल्म राम्रो नि लागेन,-1
28326,आज पनि नयाँ कोभिड पोजेटिभ नदेखिएको स्वास्थ्य तथा जनसँख्या मन्त्रालयका प्रवक्ता डा बिकास देवकोटाद्वारा जानकारी परीक्षणको सँख्या पुग्यो,1
28310,आज पनि नयाँ कोभिड पोजेटिभ नदेखिएको स्वास्थ्य तथा जनसँख्या मन्त्रालयका प्रवक्ता डा बिकास देवकोटाद्वारा जानकारी परीक्षणको सँख्या पुग्यो,-1
477,एउटा नया इतिहास लेख्यो यो फिल्म ले यती धेरै राम्रो फिल्म नेपाल मा बन्छ सक्छ भन्ने नया उदाहरण हो यो,1
230,एउटा नया इतिहास लेख्यो यो फिल्म ले यती धेरै राम्रो फिल्म नेपाल मा बन्छ सक्छ भन्ने नया उदाहरण हो यो,1


During dataset cleaning, duplicate text instances were identified and examined for annotation consistency. It was observed that some duplicate sentences had been assigned conflicting sentiment labels across different entries in the dataset. Since these instances contained contradictory annotations without a clear majority label, they were removed from the dataset rather than resolved through majority counting. This decision was made to reduce annotation ambiguity and prevent inconsistent supervision during model training.

In [7]:
conflicting = (df.groupby('Sentences')['Sentiment'].nunique())
conflicting = conflicting[conflicting > 1]
conflicting_sentences = conflicting.index
df = df[~df['Sentences'].isin(conflicting_sentences)]

In [8]:
df_clean = df.drop_duplicates()

In [9]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 35713 entries, 0 to 35788
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Sentences  35713 non-null  object
 1   Sentiment  35713 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 837.0+ KB


| Dataset Stage            | Sentences | Sentiment |
| ------------------------ | ----------------- | ------------------------------------------------------ |
| Original Dataset (df)    | 35,789            | 35,789                                                 |
| Duplicate Subset         | 113               | 113                                                    |
| Clean Dataset (df_clean) | 35,713            | 35,713                                                 |

## Topic Filtering

The data for this project has been taken from [Kaggle](https://www.kaggle.com/datasets/aayamoza/nepali-sentiment-analysis). It has been curated by taking other Nepali datasets from different sources on Kaggle and has been cleaned, organized and annotated for sentiment analysis. The dataset contains sentences that range mostly across two topics - movie reviews, Covid-19 and possibly some minor noise topics. 

An extensive portion of the dataset has been curated from the following work - [C Sitaula, A Basnet, A Mainali and TB Shahi, "Deep Learning-based Methods for Sentiment Analysis on Nepali COVID-19-related Tweets", Computational Intelligence and Neuroscience, 2021.](https://pmc.ncbi.nlm.nih.gov/articles/PMC8561567/#sec25) Although the original dataset was considered for this study, it was not publicly available and could only be accessed through direct request to the corresponding author. Therefore, the publicly available Kaggle version was used as the primary source of data.

However, since the Kaggle dataset combined multiple sentiment datasets from different domains, it also contained non-COVID-related text such as movie reviews. To ensure topic consistency and reduce domain-related variation in sentiment interpretation, keyword-based filtering was employed to isolate COVID-19-related instances for the present study.

Topic modelling has been avoided because the Sitaula et al. source dataset is COVID-19 tweets by design and also because the objective of the study is not to discover latent topics in the data but to control domain-specific variation in the task. Since sentiment is context dependent and therefore tends to vary different across different topics, restricting the dataset helps to ensure that there are no confounding effects in model training and bias evaluation. 

In [10]:
df_clean['Sentiment'].value_counts()

Sentiment
 1    15849
-1    14384
 0     5480
Name: count, dtype: int64

In [11]:
cov_key = ['कोभिड', 'कोभिड-१९', 'कोभिड १९','कोरोना', 'कोरोनाभाइरस', 'कोरोना भाइरस', 'कोरोनाले', 'कोरोना ले', 'कोरोना को', 'कोरोनाको', 
                  'कोरोना मा', 'कोरोनामा', 'कोरोना पछि', 'कोरोनाका कारण', 'कोरोना का कारण', 'कोभिडले', 'कोभिडको', 'कोभिड को', 'कोभिड मा',
                  'कोभिडमा', 'कोभिड पछि', 'लकडाउन', 'निषेधाज्ञा', 'महामारी', 'संक्रमण', 'सङ्क्रमण','संक्रमित', 'मास्क', 'खोप', 'भ्याक्सिन', 'डोज',
                  'बुस्टर', 'स्वाब टेस्ट', 'पीसीआर', 'पीसीआर टेस्ट', 'अस्पताल', 'सेनिटाइजर', 'सामाजिक दूरी', 'दूरी कायम', 'ज्वरो', 'खोकी', 'सास',
                  'लक्षण', 'बिरामी', 'कर्फ्यु', 'महामारी', 'भाइरस', 'स्वास्थ्य संकट', 'क्वारेन्टाइन', 'आइसोलेसन','जाँच']
cov_patt = r'(?:^|\s)(?:' + '|'.join(cov_key) + r')(?:\s|$)'

cov_set = df_clean[df_clean['Sentences'].str.contains(cov_patt, na = False, regex = True)]
cov_set['Sentiment'].value_counts()

Sentiment
 1    14823
-1    13430
 0     4773
Name: count, dtype: int64

In [12]:
cov_core_key = ['कोभिड', 'कोभिड-१९', 'कोभिड १९','कोरोना', 'कोरोनाभाइरस', 'कोरोना भाइरस', 'कोरोनाले', 'कोरोना ले', 'कोरोना को', 'कोरोनाको', 
                  'कोरोना मा', 'कोरोनामा', 'कोरोना पछि', 'कोरोनाका कारण', 'कोरोना का कारण', 'कोभिडले', 'कोभिडको', 'कोभिड को', 'कोभिड मा',
                  'कोभिडमा', 'कोभिड पछि', 'लकडाउन', 'निषेधाज्ञा', 'महामारी', 'संक्रमण', 'सङ्क्रमण','संक्रमित', 'मास्क', 'खोप', 'भ्याक्सिन', 'डोज',
                  'बुस्टर', 'स्वाब टेस्ट', 'पीसीआर', 'पीसीआर टेस्ट', 'सेनिटाइजर', 'सामाजिक दूरी', 'दूरी कायम', 'कर्फ्यु', 'महामारी', 'क्वारेन्टाइन', 'आइसोलेसन']
cov_core_patt = r'(?:^|\s)(?:' + '|'.join(cov_core_key) + r')(?:\s|$)'

cov_core_set = cov_set[cov_set['Sentences'].str.contains(cov_core_patt, na=False, regex = True)].copy()
cov_core_set['Sentiment'].value_counts()

Sentiment
 1    14811
-1    13421
 0     4767
Name: count, dtype: int64

Since the dataset is filtered using keywords, it's quality depends on the keywords selected. The following table shows that the keyword filtering method is reliable because there is a minimal change in class distribution when using a broader Covid keyword list vs using a stricter version that does not include any generic terms. The Covid keywords are the ones responsible for filtering the dataset and the general health related terms have minimum impact on the covid_dataset.

| Dataset              | Positive | Negative | Neutral | Total |
| -------------------- | -------- | -------- | ------- | ----- |
| covid_set (full) | 14823    | 13430    | 4774    | 33027 |
| cov_core_set         | 14811    | 13425    | 4767    | 33003 |
| Difference           | 12       | 5        | 7       | 24    |

In [13]:
cov_core_set.drop(cov_core_set[cov_core_set['Sentiment'] == 0].index,
                  inplace=True
                 )
cov_core_set = cov_core_set.reset_index(drop = True)
cov_core_set['Sentiment'].value_counts()

Sentiment
 1    14811
-1    13421
Name: count, dtype: int64

Although the original dataset contains three sentiment classes i.e., positive, negative and neutral, the current study will only use the positive and negative classes. The reason behind this choice is that neutral instances may represent ambiguous or mixed sentiment and removing it helps to maintain a clearer distinction between the other two sentiment classes. Moreover, the primary objective of the study is to identify gender bias in sentiment classification and the use of two extreme classes allows for better comparison of model predictions and thereby, acts as a reliable measurement of bias.

The dataset contains 52.46% positive labels and 47.53% negative labels. The difference between these classes is approximately 5% which does not indicate a significant class imbalance. Thus, it can be said that the dataset is relatively well distributed and can be used for training the model. 

## Preprocessing

In [14]:
def preprocess(text):
    #removing URLs if any
    text = re.sub(r'http\S+|www\.\S+', '', text)
    #removing punctuation and English words or characters if any
    text = re.sub(r'[^\u0900-\u097F+\s0-9]', '', text)
    #removing extra white spaces
    text = re.sub(r'\s+', ' ', text)
    return text

cov_core_set['Sentences'] = cov_core_set['Sentences'].apply(preprocess)

Not all stopwords are being removed because some categories contribute towards contextual meaning even though they may not be lexically relevant. Some of these are mentioned below. The value of these stopwords was evaluated using this keyword list by Pratik Shreshtha [https://github.com/prtx/Nepali-Stopwords]. 
- Words that express negation such as [न, छैन, छैनन्, होइन, होइनन्, नत्र]
- Intensifiers and degree modifiers such as [निकै, अत्यन्त, एकदम, झन, झन्, कति, साह्रै, ज्यादै, अलि, अलिकति]
- Contrastive elements such as [तर, तर पनि, भए पनि, यद्यपि, तथापि, जबकि]
- Elements that express doubt or uncertainty such as [पक्कै, अवश्य, निश्चय, सायद, शायद, होला, जस्तो लाग्छ]
- Focus/emphatic particles such as [नै, मात्र, पनि, समेत]
- Pronouns such as [म, हामी, तिमी, तिनी, उनी, कोही, कसै]
- Possessive markers: [मेरो, हाम्रो, उनको, तपाईंको]

In [15]:
stopwords = ['मा', 'को', 'का', 'ले', 'बाट', 'लागि', 'सम्म', 'बीच', 'वरिपरि', 'एक', 'एकै', 'दुई', 'तीन', 'केही', 'कुनै', 
             'र', 'तथा', 'साथै', 'प्रत्येक', 'अगाडि', 'पछि', 'तल', 'माथि', 'कृपया', 'उदाहरण', 'आदि', 'अर्थात', 'बारे', 
             'बारेमा', 'ती']

def stopword_removal(text, stopwords):
    words = text.split()
    filtered_words = [w for w in words if w not in stopwords]
    return' '.join(filtered_words)

cov_core_set['Sentences'] = cov_core_set['Sentences'].apply(lambda x: stopword_removal(x, stopwords))

In [16]:
cov_core_set.to_csv('covid19_cleaned_sentiments.csv',index = False)